# Notebook 04: MOSAIC-Med Adaptive Routing & Computational Reconciliation
### Dynamic Patient-Specific Expert Gating, Loss Regularization, and FLOP Accounting

---

### Computational Accounting Reconciliation
A critical distinction must be made between computational savings **relative to dense 4-expert ensembling** versus **relative to the monolithic baseline**:
- **Single Large Model (1$\times$2M)**: 123.9 MFLOPs
- **Heterogeneous Uniform Average**: $31.69 + 50.72 + 245.86 + 85.40 = \mathbf{413.7\text{ MFLOPs}}$
- **Dense Learned Router**: $413.7 + 1.19\text{ (router)} = \mathbf{414.9\text{ MFLOPs}}$
- **Sparse Top-2 Router**: $1.19\text{ (router)} + 296.6\text{ (mean active Top-2 pair)} = \mathbf{297.8\text{ MFLOPs}}$

**Key Insight**: Sparse Top-2 routing cuts inference computation by **28.2% relative to dense heterogeneous ensembling** (297.8 vs 414.9 MFLOPs). However, it requires **$2.40\times$ more computation than the single monolithic model** (123.9 MFLOPs).

The router objective is regularized by:
$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{diag}} + \lambda_{\text{div}} \mathcal{L}_{\text{div}} + \lambda_{\text{fail}} \mathcal{L}_{\text{fail}}$$
where $\mathcal{L}_{\text{div}} = -\mathcal{H}(\bar{w})$ prevents expert collapse by maximizing mean gating entropy.


In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('.'))

import pandas as pd
from IPython.display import Image, display

print("Modules loaded.")


## 1. Routing Strategies Comparison Table Across 5 Seeds


In [ ]:
routing_summary = pd.read_csv('results/tables/routing_multiseed_summary.csv')
print("Adaptive Routing Comparison Across 5 Seeds (Mean ± SD):")
routing_summary[['Routing Strategy', 'Active Experts', 'MFLOPs', 'Macro AUROC', 'Macro AUPRC', 'Holm p-value', 'Significant (α=0.05)']]


## 2. Learned Expert Specialization Across Clinical Conditions
Let's inspect how the router distributes weights across the 5 diagnostic classes:


In [ ]:
display(Image('results/images/adaptive_routing_analysis.png'))


## 3. Findings & Rigorous Claim Calibration
- **Descriptive Equivalence (H3)**: The Sparse Top-2 Router ($0.9113 \pm 0.0008$) achieves diagnostic parity with dense averaging ($0.9110 \pm 0.0007$) while dynamically reducing FLOPs by 28.2% relative to dense 4-expert inference.
- **Compute Trade-off**: The router enables computational pruning for multi-expert ensembles, but does NOT reduce compute relative to the single monolithic model (297.8 vs 123.9 MFLOPs).
- **Specialization**: The router routes Conduction Disturbances (CD) and Hypertrophy (HYP) predominantly to TCN-1D and MultiScale-CNN ($w_k > 0.30$), while ResNet-1D and ConvAttention-1D receive higher weight for MI and STTC.
